# Lab 08 - SVM: Hazardous Event Classification

This notebook applies **Lab 8 - SVM** to hazardous-event classification, with runtime controls for a large hourly dataset.


## Lab 8 concepts used

- Train a linear support-vector classifier.
- Compare a small-sample RBF-kernel SVM.
- Evaluate binary classification results.
- Explain why SVMs need scaling and careful sampling on larger datasets.

The primary feature set excludes `European_AQI`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'AML Assignment' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT


In [ ]:
DATASET_FILENAME = 'global_urban_smog_pm25_hourly.csv'
matches = sorted((PROJECT_ROOT / 'Datasets').glob(f'*/{DATASET_FILENAME}'))
if not matches:
    raise FileNotFoundError(f'Could not find {DATASET_FILENAME} under {PROJECT_ROOT / "Datasets"}')
DATASET_PATH = matches[0]
data = pd.read_csv(DATASET_PATH)
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values(['Timestamp', 'City']).reset_index(drop=True)
print(DATASET_PATH)
data.head()


In [ ]:
def add_time_features(df):
    out = df.copy()
    out['Timestamp'] = pd.to_datetime(out['Timestamp'])
    out = out.sort_values(['Timestamp', 'City']).reset_index(drop=True)
    out['hour'] = out['Timestamp'].dt.hour
    out['dayofweek'] = out['Timestamp'].dt.dayofweek
    out['month'] = out['Timestamp'].dt.month
    out['dayofyear'] = out['Timestamp'].dt.dayofyear
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    return out

def chronological_split(df, train_size=0.8):
    split_idx = int(len(df) * train_size)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def latest_rows(df, max_rows):
    if len(df) <= max_rows:
        return df.copy()
    return df.tail(max_rows).copy()

NUMERIC_NO_AQI = [
    'Latitude', 'Longitude', 'PM10_ug_m3', 'PM2_5_ug_m3',
    'Carbon_Monoxide_ug_m3', 'Nitrogen_Dioxide_ug_m3',
    'Ozone_ug_m3', 'Dust_ug_m3', 'UV_Index',
    'hour', 'dayofweek', 'month', 'is_weekend'
]
CATEGORICAL_FEATURES = ['City']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess, NUMERIC_NO_AQI),
        ('cat', categorical_preprocess, CATEGORICAL_FEATURES),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)


In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, precision_recall_curve, f1_score
)


In [ ]:
from sklearn.svm import LinearSVC, SVC

model_df = latest_rows(add_time_features(data), 50000)
train_df, test_df = chronological_split(model_df, train_size=0.8)
X_train = train_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_train = train_df['Hazardous_Event']
X_test = test_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_test = test_df['Hazardous_Event']

linear_svm = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LinearSVC(class_weight='balanced', C=0.5, max_iter=5000, random_state=42))
])
linear_svm.fit(X_train, y_train)
linear_pred = linear_svm.predict(X_test)
print(classification_report(y_test, linear_pred, digits=3))


In [ ]:
small_df = latest_rows(model_df, 8000)
small_train_df, small_test_df = chronological_split(small_df, train_size=0.8)
X_small_train = small_train_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_small_train = small_train_df['Hazardous_Event']
X_small_test = small_test_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_small_test = small_test_df['Hazardous_Event']

rbf_svm = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=42))
])
rbf_svm.fit(X_small_train, y_small_train)
rbf_pred = rbf_svm.predict(X_small_test)
print(classification_report(y_small_test, rbf_pred, digits=3))


In [ ]:
comparison = pd.DataFrame([
    {
        'model': 'LinearSVC',
        'rows_used': len(model_df),
        'balanced_accuracy': balanced_accuracy_score(y_test, linear_pred),
        'f1_hazardous': f1_score(y_test, linear_pred),
    },
    {
        'model': 'RBF SVC small sample',
        'rows_used': len(small_df),
        'balanced_accuracy': balanced_accuracy_score(y_small_test, rbf_pred),
        'f1_hazardous': f1_score(y_small_test, rbf_pred),
    },
])
comparison


In [ ]:
cm = confusion_matrix(y_test, linear_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Not hazardous', 'Hazardous'],
            yticklabels=['Not hazardous', 'Hazardous'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Linear SVM confusion matrix')
plt.show()


## What was learned from Lab 8

SVMs are sensitive to feature scale and can become expensive on large datasets. A linear SVM is a practical baseline for the full lab subset, while RBF SVMs should be tested on controlled samples before being considered for the final pipeline.
